In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

def time_vector(n_samples, duration=1.0):
    """Evenly spaced time samples on [0, duration), like np.linspace(..., endpoint=False).

    torch.linspace has no endpoint=False, so we take n_samples + 1 points and drop
    the last. dtype is float64 to match NumPy's default precision -- important for
    square_wave, whose sign() flips at zero-crossings if the samples differ by a ULP.
    """
    return torch.linspace(0.0, duration, n_samples + 1, device=device, dtype=torch.float64)[:-1]

In [ ]:
def square_wave(t, f0=1.0):
    """Ideal square wave evaluated at the time samples in `t`.

    PyTorch version of the NumPy `square_wave` from Square_wave_harmonics.ipynb:
        np.sign(np.sin(2.0 * np.pi * f0 * t))

    Returns a tensor the same shape / dtype / device as `t`, with values in
    {-1, 0, +1}: the sign of sin(2*pi*f0*t). This is the target that the
    truncated Fourier series (`square_wave_fourier`) converges toward.
    """
    phase = 2 * torch.pi * f0 * t      # same shape as t
    sinwave = torch.sin(phase)         # same shape as t
    result = torch.sign(sinwave)       # same shape as t, values in {-1, 0, 1}
    return result

In [ ]:
def square_wave_fourier(t, f0, num_harmonics):
    """Truncated Fourier-series approximation of a square wave.

    PyTorch version of the NumPy `square_wave_fourier` from
    Square_wave_harmonics.ipynb. Sums the first `num_harmonics` ODD harmonics:

        (4/pi) * sum_{k=0}^{num_harmonics-1}  sin(2*pi*n*f0*t) / n,
        where n = 2*k + 1   (n = 1, 3, 5, ...)

    The NumPy version does this with a Python `for k in range(num_harmonics)`
    loop. Here it is vectorised with broadcasting so every harmonic is computed
    at once and the whole thing runs on GPU when `t` is on GPU.

    Returns a 1-D tensor of shape (len(t),), same dtype / device as `t`.
    """
    # Step 1: odd harmonic numbers n = 1, 3, 5, ...
    n = torch.arange(1, 2 * num_harmonics, 2, device=t.device, dtype=t.dtype)  # shape (num_harmonics,)

    # Step 2 & 3: reshape to broadcast, then compute every harmonic at once
    n_col = n[:, None]                  # shape (num_harmonics, 1)
    t_row = t[None, :]                  # shape (1, len(t))
    terms = torch.sin(2 * torch.pi * n_col * f0 * t_row) / n_col   # shape (num_harmonics, len(t))

    # Step 4: sum over harmonics, scale by 4/pi
    result = (4.0 / torch.pi) * terms.sum(dim=0)   # shape (len(t),)
    return result

testing time

In [ ]:
# Original NumPy implementations from Square_wave_harmonics.ipynb, kept here
# under _np names so we can check the PyTorch versions against them.

def square_wave_np(t, f0=1.0):
    return np.sign(np.sin(2.0 * np.pi * f0 * t))

def square_wave_fourier_np(t, f0, num_harmonics):
    result = np.zeros_like(t)
    for k in range(num_harmonics):
        n = 2 * k + 1  # square wave has only odd harmonics
        result += np.sin(2 * np.pi * n * f0 * t) / n
    return (4 / np.pi) * result

In [ ]:
# Same inputs for both implementations
N = 2048
f0 = 1.0
num_harmonics = 5

# PyTorch version (on `device`, then bring back to CPU/NumPy for comparison)
t_torch = time_vector(N, duration=1.0)
y_torch = square_wave_fourier(t_torch, f0, num_harmonics).cpu().numpy()
sq_torch = square_wave(t_torch, f0).cpu().numpy()

# NumPy reference version, matching time vector (np.linspace(..., endpoint=False))
t_np = np.linspace(0.0, 1.0, N, endpoint=False)
y_np = square_wave_fourier_np(t_np, f0, num_harmonics)
sq_np = square_wave_np(t_np, f0)

print('square_wave_fourier matches NumPy:', np.allclose(y_torch, y_np, atol=1e-5))  # expect True
print('square_wave matches NumPy:        ', np.allclose(sq_torch, sq_np, atol=1e-5))  # expect True

plt.figure(figsize=(10, 4))
plt.plot(t_np, sq_np, 'k', alpha=0.5, label='square wave')
plt.plot(t_np, y_np, 'r--', label='NumPy Fourier')
plt.plot(t_np, y_torch, 'b:', label='PyTorch Fourier')
plt.title(f'Square wave Fourier series, num_harmonics={num_harmonics}')
plt.ylim(-1.5, 1.5)
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
def naive_dft(x):
    """..."""
    N = x.shape[-1]

    # Step 1: index vector 0..N-1
    idx = torch.arange(N, device=x.device, dtype=torch.float64)

    # Step 2: outer product k*n
    kn = idx[:, None] * idx[None, :]           # shape (N, N)

    # Step 3: angle matrix
    angle = -2 * torch.pi * kn / N              # shape (N, N)

    # Step 4: DFT matrix W = exp(-2j*pi*k*n/N)
    W = torch.polar(torch.ones_like(angle), angle)   # complex128, shape (N, N)

    # Step 5: matmul
    x_complex = x.to(W.dtype)                   # cast x to complex to match W
    return W @ x_complex

In [ ]:
def naive_dft_gpu(x):
    """..."""
    x_gpu = x.to('cuda')
    N = x_gpu.shape[-1]

    idx = torch.arange(N, device=x_gpu.device, dtype=torch.float64)
    kn = idx[:, None] * idx[None, :]
    angle = -2 * torch.pi * kn / N
    W = torch.polar(torch.ones_like(angle), angle)

    x_gpu_complex = x_gpu.to(W.dtype)
    return W @ x_gpu_complex

In [ ]:
N_test = 512
t_test = time_vector(N_test)
y_test = square_wave_fourier(t_test, 1.0, 5)

X_naive = naive_dft(y_test)
X_fft = torch.fft.fft(y_test.to(torch.complex128))
print('naive_dft matches torch.fft.fft:', torch.allclose(X_naive, X_fft.cpu, atol=1e-6))

# only run this part on Rangpur with a GPU node -- will error on your laptop
if torch.cuda.is_available():
    X_gpu = naive_dft_gpu(y_test)
    print('naive_dft_gpu matches torch.fft.fft:', torch.allclose(X_gpu.cpu(), X_fft, atol=1e-6))
else:
    print('No GPU available here -- test naive_dft_gpu on Rangpur.')

In [ ]:
import time

def naive_dft_np(x):
    """Original NumPy nested-loop DFT -- O(N^2) with Python-level loops."""
    N = len(x)
    X = np.zeros(N, dtype=complex)
    for k in range(N):
        for n in range(N):
            X[k] += x[n] * np.exp(-2j * np.pi * k * n / N)
    return X


def time_it(fn, *args, n_repeat=3, cuda=False):
    """Best-of-n timing. CUDA kernels launch asynchronously, so we synchronise
    before starting and before stopping -- otherwise we'd time the launch call,
    not the actual GPU work."""
    if cuda:
        torch.cuda.synchronize()
    best = float('inf')
    for _ in range(n_repeat):
        t0 = time.perf_counter()
        fn(*args)
        if cuda:
            torch.cuda.synchronize()
        best = min(best, time.perf_counter() - t0)
    return best


SIZES = [128, 256, 512, 1024]

print(f"{'N':>6} {'numpy loop':>14} {'torch CPU':>12} {'torch GPU':>12} {'FFT':>10}")
print("-" * 58)

for N in SIZES:
    t = time_vector(N)
    y = square_wave_fourier(t, 1.0, 5)
    y_cpu = y.cpu()
    y_np = y_cpu.numpy()

    # n_repeat=1 for the NumPy loop -- it's O(N^2) in pure Python and painfully slow
    t_np    = time_it(naive_dft_np, y_np, n_repeat=1)
    t_torch = time_it(naive_dft, y_cpu)
    t_gpu   = time_it(naive_dft_gpu, y_cpu, cuda=True) if torch.cuda.is_available() else float('nan')
    t_fft   = time_it(torch.fft.fft, y_cpu.to(torch.complex128))

    print(f"{N:>6} {t_np*1e3:>12.2f}ms {t_torch*1e3:>10.2f}ms "
          f"{t_gpu*1e3:>10.2f}ms {t_fft*1e3:>8.3f}ms")